In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df=pd.read_csv(r"C:\Users\shaha\Downloads\spam.csv")

In [3]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

In [5]:
import nltk
import string
punctuation=string.punctuation
nltk.download("stopwords")
from nltk.corpus import stopwords
sp=set(stopwords.words("english"))
nltk.download("punkt")
lemmatizer = WordNetLemmatizer()
def clean(text):
    words=word_tokenize(text)
    words=[word for word in words if word not in sp and word not in punctuation]
    words=[lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shaha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shaha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
import re

In [7]:
def preprocess_text(text): 
    text=text.lower()
    text=re.sub('\[.*?\]','',text)
    text=re.sub("\\W"," ",text)
    text=re.sub('https?://\S+|www\.\S+','',text)
    text=re.sub('<.*?>+','',text)
    text=re.sub('[%s]'% re.escape(string.punctuation),'',text)
    text=re.sub('\n','',text)
    text=re.sub('\w*\d\w*','',text)
    return text

In [8]:
df['Message']=df['Message'].apply(preprocess_text)
df['Message']=df['Message'].apply(clean)

In [9]:
df

,Category,Message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni
2,spam,free entry wkly comp win fa cup final tkts may...
3,ham,u dun say early hor u c already say
4,ham,nah think go usf life around though
...,...,...
5567,spam,time tried contact u u pound prize claim easy ...
5568,ham,ü b going esplanade fr home
5569,ham,pity mood suggestion
5570,ham,guy bitching acted like interested buying some...


In [10]:
from sklearn.preprocessing import LabelEncoder

In [11]:
le=LabelEncoder()

In [12]:
df['Category']=le.fit_transform(df['Category'])

In [13]:
df.columns

Index(['Category', 'Message'], dtype='object')

In [14]:
x=df['Message']

In [15]:
x

0       go jurong point crazy available bugis n great ...
1                                 ok lar joking wif u oni
2       free entry wkly comp win fa cup final tkts may...
3                     u dun say early hor u c already say
4                     nah think go usf life around though
                              ...                        
5567    time tried contact u u pound prize claim easy ...
5568                          ü b going esplanade fr home
5569                                 pity mood suggestion
5570    guy bitching acted like interested buying some...
5571                                       rofl true name
Name: Message, Length: 5572, dtype: object

In [16]:
y=df['Category']

In [17]:
y

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: Category, Length: 5572, dtype: int64

In [18]:
import tensorflow as tf

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [21]:
x_train.shape

(4457,)

In [22]:
y_train.shape

(4457,)

In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer
max_vocab = 10000
tokenizer = Tokenizer(num_words=max_vocab)
tokenizer.fit_on_texts(x_train)

In [24]:
x_train=tokenizer.texts_to_sequences(x_train)
x_test=tokenizer.texts_to_sequences(x_test)

In [25]:
x_train=tf.keras.preprocessing.sequence.pad_sequences(x_train,padding='post',maxlen=256)
x_test =tf.keras.preprocessing.sequence.pad_sequences(x_test, padding='post', maxlen=256)

In [26]:
max_len=256
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_vocab, output_dim=32, input_length=max_len),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1)
])
model.build(input_shape=(None, max_len))
model.summary()

C:\Users\shaha\Downloads\ANACONDA\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 256, 32)             │         320,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 256, 128)            │          49,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 32)                  │          18,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 390,401 (1.49 MB)

 Trainable params: 390,401 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [28]:
history = model.fit(x_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 88s 681ms/step - accuracy: 0.9161 - loss: 0.2748 - val_accuracy: 0.9697 - val_loss: 0.1629
Epoch 2/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 68s 607ms/step - accuracy: 0.9885 - loss: 0.0697 - val_accuracy: 0.9776 - val_loss: 0.2116
Epoch 3/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 73s 653ms/step - accuracy: 0.9944 - loss: 0.0405 - val_accuracy: 0.9798 - val_loss: 0.2545
Epoch 4/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 71s 632ms/step - accuracy: 0.9972 - loss: 0.0372 - val_accuracy: 0.9776 - val_loss: 0.2394
Epoch 5/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 69s 610ms/step - accuracy: 0.9975 - loss: 0.0367 - val_accuracy: 0.9787 - val_loss: 0.2512
Epoch 6/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 72s 644ms/step - accuracy: 0.9978 - loss: 0.0320 - val_accuracy: 0.9720 - val_loss: 0.2362
Epoch 7/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 69s 617ms/step - accuracy: 0.9978 - loss: 0.0322 - val_accuracy: 0.9652 - val_loss: 0.3818
Epoch 8/10
112/112 ━━━━━━━━━━━━━━━━━━━━ 68s 612ms/step - accuracy: 0.9972 - loss: 0

In [29]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 154ms/step - accuracy: 0.9848 - loss: 0.1721
Test Accuracy: 98.48%
